In [2]:
%load_ext autoreload
%autoreload 2

import itertools
from math import prod

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


from assignment_1.modules.models import fit_bpr, bpr_scorer
from assignment_1.modules.util import (
    calc_pair_accuracy,
    load_csv,
    random_split,
    make_pair_validation,
)

DATA_DIR = "../data"  # notebook runs from notebooks/

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
TASK_TRAIN_FILE = "tasks_3_4_train.csv"
TASK_TEST_FILE = "task_4_test.csv"
train = load_csv(TASK_TRAIN_FILE, DATA_DIR)
test = load_csv(TASK_TEST_FILE, DATA_DIR)

In [5]:
train.head(5)

,user,item,timestamp,rating
0,u1838647363,i0846907466,1360914337,3
1,u7093796076,i2832259763,1472021070,4
2,u4779464116,i9244553928,1480231213,5
3,u2107634551,i2346931943,1343026318,4
4,u0480528309,i3693294311,1405148743,4


In [6]:
test.head(5)

,user,timestamp,item1,item2,which
0,u9337049416,1455001830,i0277476004,i3096330769,NaN
1,u0128064500,1405148935,i1691394674,i7145553690,NaN
2,u2287415013,1488186442,i7337593393,i6009132051,NaN
3,u2406803247,1346135928,i3019728190,i8275841250,NaN
4,u1146240669,1312185505,i8121318922,i9624491588,NaN


In [ ]:
first_seen = train.groupby("item")["timestamp"].min()     # earliest interaction per book
t = test["timestamp"]
early1 = test["item1"].map(first_seen) > t                # item1 appears only after this row's time
early2 = test["item2"].map(first_seen) > t
print(f"item1 'not yet seen' at t: {early1.mean():.2%}, item2: {early2.mean():.2%}, "
      f"exactly one: {(early1 ^ early2).mean():.2%}")

item1 'not yet seen' at t: 12.97%, item2: 13.56%, exactly one: 25.47%


In [7]:
tr, pairs = make_pair_validation(train, include_timestamp=True, seed=0)

first_seen = tr.groupby("item")["timestamp"].min()
t = pairs["timestamp"]
early1 = pairs["item1"].map(first_seen) > t
early2 = pairs["item2"].map(first_seen) > t
one = early1 ^ early2                                   # exactly one item "not yet seen"
print(f"exactly one not-yet-seen: {one.mean():.2%}   (test: 25.47%)")

# within those rows: is the already-seen item the real one?
seen_item = np.where(early1, pairs["item2"], pairs["item1"])
print(f"rule accuracy on those rows: {(seen_item[one] == pairs['which'][one]).mean():.3f}")

exactly one not-yet-seen: 26.59%   (test: 25.47%)
rule accuracy on those rows: 0.970


In [8]:
tr, pairs = make_pair_validation(train, seed=0, include_timestamp=True)
calc_pair_accuracy(pairs, bpr_scorer(fit_bpr(tr, factors=5, regularization=0.01, learning_rate=0.01, iterations=100)))

pairs.head()

,user,item1,item2,which,timestamp
0,u8203075739,i1096367962,i3966153512,i1096367962,1500357411
1,u0999151525,i5186851769,i0169788189,i5186851769,1453107656
2,u6752727348,i7416016481,i1616614560,i7416016481,1448523563
3,u2237075147,i0004924020,i7351848776,i7351848776,1400055468
4,u6268264360,i6007667239,i3047831401,i3047831401,1412151117


In [10]:
fit = fit_bpr(tr, factors=5, regularization=0.01, learning_rate=0.01, iterations=100)
score = bpr_scorer(fit)
base = calc_pair_accuracy(pairs, bpr_scorer(fit))   

users = pairs["user"].to_numpy()
s1 = score(users, pairs["item1"].to_numpy())
s2 = score(users, pairs["item2"].to_numpy())
bpr_pred = np.where(s1 >= s2, pairs["item1"], pairs["item2"])       # BPR's choice per row

which = pairs["which"].to_numpy()
print(f"BPR overall:        {(bpr_pred == which).mean():.4f}")
print(f"BPR on rule rows:   {(bpr_pred[one] == which[one]).mean():.4f}")   # what the rule (0.970) has to beat
combined = np.where(one, seen_item, bpr_pred)
print(f"combined:           {(combined == which).mean():.4f}")

BPR overall:        0.8161
BPR on rule rows:   0.8647
combined:           0.8442


In [11]:
def run_grid_t4(grid: dict) -> pd.DataFrame:
    which = pairs["which"].to_numpy()
    users = pairs["user"].to_numpy()
    rows = []
    for values in tqdm(itertools.product(*grid.values()), total=prod(map(len, grid.values()))):
        params = dict(zip(grid, values))
        score = bpr_scorer(fit_bpr(tr, **params))
        s1 = score(users, pairs["item1"].to_numpy())
        s2 = score(users, pairs["item2"].to_numpy())
        bpr_pred = np.where(s1 >= s2, pairs["item1"], pairs["item2"])
        combined = np.where(one, seen_item, bpr_pred)
        rows.append({**params,
                     "bpr_acc": (bpr_pred == which).mean(),
                     "combined_acc": (combined == which).mean()})
    return pd.DataFrame(rows).sort_values("combined_acc", ascending=False)


In [12]:
grid = {
    "factors":        [5, 20, 50],
    "regularization": [0.001, 0.01, 0.1],
    "learning_rate":  [0.01],
    "iterations":     [100],
}

results1 = run_grid_t4(grid)

results1

  0%|          | 0/9 [00:00<?, ?it/s]

,factors,regularization,learning_rate,iterations,bpr_acc,combined_acc
7,50,0.010,0.01,100,0.861255,0.881245
4,20,0.010,0.01,100,0.852915,0.874136
6,50,0.001,0.01,100,0.837193,0.865068
3,20,0.001,0.01,100,0.839677,0.864543
1,5,0.010,0.01,100,0.816133,0.844204
0,5,0.001,0.01,100,0.815028,0.843637
5,20,0.100,0.01,100,0.776574,0.811613
8,50,0.100,0.01,100,0.773208,0.808828
2,5,0.100,0.01,100,0.769332,0.804484


In [13]:

grid = {
    "factors":        [50, 100, 200],
    "regularization": [0.003, 0.01, 0.03],
    "learning_rate":  [0.01],
    "iterations":     [100],
}

results2 = run_grid_t4(grid)

results2

  0%|          | 0/9 [00:00<?, ?it/s]

,factors,regularization,learning_rate,iterations,bpr_acc,combined_acc
7,200,0.010,0.01,100,0.862416,0.881937
4,100,0.010,0.01,100,0.862011,0.881804
1,50,0.010,0.01,100,0.861255,0.881245
6,200,0.003,0.01,100,0.856756,0.879754
3,100,0.003,0.01,100,0.854566,0.878026
0,50,0.003,0.01,100,0.850648,0.874619
2,50,0.030,0.01,100,0.840300,0.863228
5,100,0.030,0.01,100,0.838851,0.862151
8,200,0.030,0.01,100,0.837704,0.860982
